# Robustness checks

This notebook reproduces, from the disclosure-safe tables in `data/derived/`, the numbers reported for the supervised text model, the text-measure diagnostics, the cross-format comparison, and the sensitivity analyses in the main text and SI Appendix. Review-level records are not required; the estimates were computed from the restricted review-level data and are released here as aggregates.

Run every cell from top to bottom. Each section prints the released estimates and then checks the corresponding statements in the paper (✓ holds, ✗ does not).

## Setup

Paths, group labels, and small helpers for tables and checks. The next cell finds the replication-package root even when the Jupyter working directory is `code/`. Only the Python standard library is used.

In [1]:
from __future__ import annotations

import csv
import json
import math
from pathlib import Path


starts = [Path.cwd().resolve()]
try:
    # VS Code / Cursor expose the notebook path even when cwd is code/.
    starts.insert(0, Path(__vsc_ipynb_file__).resolve().parent)
except NameError:
    pass

ROOT = None
seen = set()
for start in starts:
    for p in [start, *start.parents]:
        if str(p) in seen:
            continue
        seen.add(str(p))
        if (p / "data" / "derived").is_dir():
            ROOT = p
            break
        nested = p / "rating-comment-gaps"
        if (nested / "data" / "derived").is_dir():
            ROOT = nested
            break
    if ROOT is not None:
        break
if ROOT is None:
    raise FileNotFoundError(
        "Could not locate data/derived. Open this notebook from the replication package."
    )

DATA = ROOT / "data" / "derived"
SUMMARY = json.loads((DATA / "main_results_summary.json").read_text(encoding="utf-8"))
GROUPS = [
    "White_Male", "White_Female",
    "Black_Male", "Black_Female",
    "Hispanic_Male", "Hispanic_Female",
    "Asian_Male", "Asian_Female",
]
SHORT = {
    "White_Male": "White men", "White_Female": "White women",
    "Black_Male": "Black men", "Black_Female": "Black women",
    "Hispanic_Male": "Hispanic men", "Hispanic_Female": "Hispanic women",
    "Asian_Male": "Asian men", "Asian_Female": "Asian women",
}


def rows(name):
    with (DATA / name).open(newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def num(x):
    return float(x) if x not in ("", None) else float("nan")


def table(records, columns, widths=None):
    """Print a plain-text table; columns is a list of (header, function of a record)."""
    cells = [[str(f(r)) for _, f in columns] for r in records]
    widths = widths or [max([len(h)] + [len(row[i]) for row in cells]) + 2 for i, (h, _) in enumerate(columns)]
    print("  ".join(h.ljust(w) for (h, _), w in zip(columns, widths)))
    for row in cells:
        print("  ".join(v.ljust(w) for v, w in zip(row, widths)))


def check(statement, condition):
    """Print whether a statement made in the paper holds in the released estimates."""
    print(("\u2713 " if condition else "\u2717 ") + statement)


def pearson(x, y):
    mx, my = sum(x) / len(x), sum(y) / len(y)
    sxy = sum((a - mx) * (b - my) for a, b in zip(x, y))
    return sxy / math.sqrt(sum((a - mx) ** 2 for a in x) * sum((b - my) ** 2 for b in y))


def ranks(x):
    order = sorted(range(len(x)), key=lambda i: x[i])
    r = [0.0] * len(x)
    for rank, i in enumerate(order, 1):
        r[i] = float(rank)
    return r

## Analysis samples

The four nested samples (SI Appendix, *Analysis Samples*): full star sample (A), star text sample (B), binary sample (C), and binary text sample (D). Instructor counts largely determine the precision of each group's estimates, because standard errors are clustered by instructor.

In [2]:
counts = rows("si_sample_counts.csv")
for sample in ["A. Full star sample", "B. Star text sample", "C. Binary sample", "D. Binary text sample"]:
    total = next(r for r in counts if r["sample"] == sample and r["group"] == "Total")
    print(f"{sample:24s} {int(total['reviews']):>10,d} reviews  {int(total['instructors']):>8,d} instructors")
print()
table([r for r in counts if r["sample"] == "D. Binary text sample" and r["group"] != "Total"],
      [("Group (sample D)", lambda r: SHORT[r["group"]]), ("Reviews", lambda r: f"{int(r['reviews']):,}"),
       ("Instructors", lambda r: f"{int(r['instructors']):,}")])
print()
check("Sample A has 4,050,824 reviews of 385,161 instructors",
      SUMMARY["clean_star_n"] == 4050824 and SUMMARY["clean_star_instructors"] == 385161)
check("Sample B has 3,304,978 reviews; sample D has 428,386",
      SUMMARY["star_text_n"] == 3304978 and SUMMARY["wta_text_n"] == 428386)

A. Full star sample       4,050,824 reviews   385,161 instructors
B. Star text sample       3,304,978 reviews   369,411 instructors
C. Binary sample            487,110 reviews   138,840 instructors
D. Binary text sample       428,386 reviews   131,123 instructors

Group (sample D)    Reviews    Instructors  
White men           210,584    63,330       
White women         148,341    48,035       
Black men           1,888      551          
Black women         1,105      345          
Hispanic men        20,304     5,529        
Hispanic women      16,310     4,808        
Asian men           16,939     4,903        
Asian women         12,915     3,622        

✓ Sample A has 4,050,824 reviews of 385,161 instructors
✓ Sample B has 3,304,978 reviews; sample D has 428,386


## Supervised text model

A ridge regression on all words and two-word phrases in the full comment, trained on ratings net of each group's mean with instructor-grouped cross-fitting (SI Appendix, *Supervised Text Model*). Its out-of-fold prediction replaces the seven-measure index in the gap model. Star results use sample B and binary results use sample D.

In [3]:
sup = rows("si_supervised_decomposition.csv")
for outcome in ["Star rating (SD)", "Would take again (percentage points)"]:
    print(outcome)
    d = [r for r in sup if r["outcome"] == outcome and r["group"] != "White_Male"]
    table(d, [("Group", lambda r: SHORT[r["group"]]),
              ("Observed", lambda r: f"{num(r['observed']):+.3f}"),
              ("Predicted from text", lambda r: f"{num(r['text_predicted']):+.3f}"),
              ("Additional gap", lambda r: f"{num(r['additional']):+.3f}"),
              ("95% CI", lambda r: f"[{num(r['additional_lower']):+.3f}, {num(r['additional_upper']):+.3f}]")])
    print()

star = {r["group"]: r for r in sup if r["outcome"].startswith("Star") and r["group"] != "White_Male"}
wta = {r["group"]: r for r in sup if r["outcome"].startswith("Would") and r["group"] != "White_Male"}
print(f"Within-R^2: seven measures {SUMMARY['within_r2_seven_measures']:.2f}, supervised {SUMMARY['within_r2_supervised']:.2f}")
add = [num(r["additional"]) for r in star.values()]
check("Star rating: every additional gap stays negative, from about 0.01 to 0.10 SD",
      all(num(r["additional_upper"]) < 0 for r in star.values()) and round(-max(add), 2) == 0.01 and round(-min(add), 2) == 0.10)
women = ["White_Female", "Hispanic_Female", "Asian_Female"]
others = ["White_Male", "Black_Male", "Black_Female", "Hispanic_Male", "Asian_Male"]
check("Binary response: significant for White, Hispanic, and Asian women (1.0 to 1.7 points)",
      all(num(wta[g]["additional_upper"]) < 0 for g in women)
      and sorted(round(-num(wta[g]["additional"]), 1) for g in women)[0] == 1.0
      and sorted(round(-num(wta[g]["additional"]), 1) for g in women)[-1] == 1.7)
check("Binary response: not significant for men or for Black women",
      all(num(wta[g]["additional_lower"]) <= 0 <= num(wta[g]["additional_upper"]) for g in others if g in wta))

Star rating (SD)
Group             Observed    Predicted from text    Additional gap    95% CI            
White women       -0.034      -0.021                 -0.014            [-0.015, -0.012]  
Black men         -0.288      -0.189                 -0.099            [-0.114, -0.084]  
Black women       -0.284      -0.203                 -0.080            [-0.100, -0.061]  
Hispanic men      -0.095      -0.058                 -0.037            [-0.041, -0.032]  
Hispanic women    -0.098      -0.058                 -0.040            [-0.045, -0.035]  
Asian men         -0.171      -0.098                 -0.073            [-0.078, -0.067]  
Asian women       -0.154      -0.088                 -0.066            [-0.072, -0.060]  

Would take again (percentage points)
Group             Observed    Predicted from text    Additional gap    95% CI            
White women       -2.510      -0.830                 -1.680            [-1.888, -1.473]  
Black men         -8.725      -9.488         

### Text weights estimated with unpenalized group indicators

The supervised model re-estimated with the race-by-gender indicators included in the ridge regression and left unpenalized, so that the text weights reflect only how wording relates to ratings within groups. The fitted weights are applied to each review's comment, so group differences in what students wrote count toward the text-predicted gap (SI Appendix, *Supervised Text Model*). Features, folds, and penalty are unchanged.

In [4]:
unp = rows("si_supervised_unpenalized_groups.csv")
sup_by = {(r["outcome"], r["group"]): r for r in sup}
for outcome in ["Star rating (SD)", "Would take again (percentage points)"]:
    print(outcome)
    d = [r for r in unp if r["outcome"] == outcome and r["group"] != "White_Male"]
    table(d, [("Group", lambda r: SHORT[r["group"]]),
              ("Additional (main)", lambda r: f"{num(sup_by[(r['outcome'], r['group'])]['additional']):+.3f}"),
              ("Additional (unpenalized)", lambda r: f"{num(r['additional']):+.3f}"),
              ("95% CI", lambda r: f"[{num(r['additional_lower']):+.3f}, {num(r['additional_upper']):+.3f}]"),
              ("Change", lambda r: f"{num(r['additional']) - num(sup_by[(r['outcome'], r['group'])]['additional']):+.3f}")])
    print()

ustar = {r["group"]: r for r in unp if r["outcome"].startswith("Star") and r["group"] != "White_Male"}
uwta = {r["group"]: r for r in unp if r["outcome"].startswith("Would") and r["group"] != "White_Male"}
change = {g: num(ustar[g]["additional"]) - num(star[g]["additional"]) for g in ustar}
largest = sorted(change, key=lambda g: abs(change[g]), reverse=True)[:2]
print(f"Within-R^2: main supervised {SUMMARY['within_r2_supervised']:.2f}, unpenalized group indicators {SUMMARY['within_r2_supervised_unpenalized_groups']:.2f}")
check(f"Star rating: every additional gap becomes slightly smaller, by at most 0.010 SD (largest change {max(map(abs, change.values())):.4f})",
      all(v > 0 for v in change.values()) and round(max(map(abs, change.values())), 3) <= 0.010)
check("Star rating: the largest changes are for Asian men and women", set(largest) == {"Asian_Male", "Asian_Female"})
check("Star rating: every additional gap remains negative and significant", all(num(r["additional_upper"]) < 0 for r in ustar.values()))
check("Within-R^2 is unchanged at 0.71",
      round(SUMMARY["within_r2_supervised_unpenalized_groups"], 2) == round(SUMMARY["within_r2_supervised"], 2) == 0.71)
check("Binary response: White, Hispanic, and Asian women again retain significant additional gaps",
      all(num(uwta[g]["additional_upper"]) < 0 for g in women))
check("Binary response: still not significant for men or for Black women",
      all(num(uwta[g]["additional_lower"]) <= 0 <= num(uwta[g]["additional_upper"]) for g in others if g in uwta))


Star rating (SD)
Group             Additional (main)    Additional (unpenalized)    95% CI              Change  
White women       -0.014               -0.012                      [-0.014, -0.011]    +0.001  
Black men         -0.099               -0.092                      [-0.107, -0.077]    +0.007  
Black women       -0.080               -0.077                      [-0.096, -0.057]    +0.003  
Hispanic men      -0.037               -0.035                      [-0.039, -0.030]    +0.002  
Hispanic women    -0.040               -0.037                      [-0.042, -0.032]    +0.003  
Asian men         -0.073               -0.063                      [-0.068, -0.057]    +0.010  
Asian women       -0.066               -0.056                      [-0.063, -0.050]    +0.010  

Would take again (percentage points)
Group             Additional (main)    Additional (unpenalized)    95% CI              Change  
White women       -1.680               -1.639                      [-1.846, -1.43

## Text-measure diagnostics

Collinearity among the seven gender-neutralized text measures, and the agreement between the star and binary joint weights (SI Appendix, *Text-Association Models*). The weight comparison is computed directly from the joint coefficients in `figure1_a.csv`.

In [5]:
vif = rows("si_text_measure_vif.csv")
table(vif, [("Text measure", lambda r: r["dimension"]), ("VIF (masked)", lambda r: f"{num(r['vif_masked']):.2f}"),
            ("VIF (original)", lambda r: f"{num(r['vif_original']):.2f}")])
cor = {r["measure"]: r for r in rows("si_text_measure_correlations.csv")}
pairs = [("Warmth", "Competence"), ("Warmth", "Effort"), ("Competence", "Effort")]
pc = [num(cor[a][b]) for a, b in pairs]
print("\nCorrelations among warmth, competence, and effort:", ", ".join(f"{v:.3f}" for v in pc))

joint = [r for r in rows("figure1_a.csv") if r["specification"] == "Joint (all seven)"]
ws = [num(r["estimate"]) for r in joint if r["outcome"] == "Star rating"]
ww = [num(r["estimate"]) for r in joint if r["outcome"] == "Would take again"]
cosine = sum(a * b for a, b in zip(ws, ww)) / math.sqrt(sum(a * a for a in ws) * sum(b * b for b in ww))
print(f"Star vs binary joint weights: r = {pearson(ws, ww):.3f}, Spearman = {pearson(ranks(ws), ranks(ww)):.3f}, cosine = {cosine:.3f}\n")

v = {r["dimension"]: num(r["vif_masked"]) for r in vif}
check("Pairwise correlations of warmth, competence, and effort run from 0.84 to 0.88",
      round(min(pc), 2) == 0.84 and round(max(pc), 2) == 0.88)
check("VIFs are 5.0, 6.6, and 5.6 for warmth, competence, and effort; 1.55 and 1.24 for brilliance and complaint severity",
      [round(v[k], 1) for k in ["Warmth", "Competence", "Effort"]] == [5.0, 6.6, 5.6]
      and [round(v[k], 2) for k in ["Brilliance", "Complaint severity"]] == [1.55, 1.24])
check("All VIFs are below 10", max(v.values()) < 10)
check("Star and binary weights agree: r = 0.993, Spearman 1.000, cosine 0.995",
      round(pearson(ws, ww), 3) == 0.993 and round(pearson(ranks(ws), ranks(ww)), 3) == 1.0 and round(cosine, 3) == 0.995)

Text measure          VIF (masked)    VIF (original)  
Warmth                4.96            4.31            
Competence            6.65            5.80            
Brilliance            1.55            1.63            
Effort                5.62            4.80            
Complaint severity    1.24            1.18            
Positive affect       1.36            1.34            
Negative affect       1.25            1.25            

Correlations among warmth, competence, and effort: 0.855, 0.841, 0.881
Star vs binary joint weights: r = 0.993, Spearman = 1.000, cosine = 0.995

✓ Pairwise correlations of warmth, competence, and effort run from 0.84 to 0.88
✓ VIFs are 5.0, 6.6, and 5.6 for warmth, competence, and effort; 1.55 and 1.24 for brilliance and complaint severity
✓ All VIFs are below 10
✓ Star and binary weights agree: r = 0.993, Spearman 1.000, cosine 0.995


## Text-association models with alternative scores

The models behind Fig. 1A, re-estimated on the same samples and specification (race-by-gender indicators, controls, and fixed effects) with two alternative sets of text scores: the original embedding scores before masking names and gendered terms, and LIWC-22 positive and negative emotion words in place of positive and negative tone (SI Appendix, *Text-Association Models*). The masked estimates are those in `figure1_a.csv`.

In [6]:
alt = rows("si_text_weights_alternative_scores.csv")
main = rows("figure1_a.csv")
ORDER = ["Warmth", "Competence", "Brilliance", "Effort", "Complaint severity", "Positive affect", "Negative affect"]
EMO_NAME = {"Positive affect": "Positive emotion", "Negative affect": "Negative emotion"}
SCORES = {
    "Masked (main)": (main, lambda d: d),
    "Original": ([r for r in alt if r["scores"].startswith("Original")], lambda d: d),
    "Emotion words": ([r for r in alt if r["scores"].startswith("LIWC")], lambda d: EMO_NAME.get(d, d)),
}


def weights(scores, outcome, spec="Joint (all seven)"):
    data, name = SCORES[scores]
    est = {r["dimension"]: num(r["estimate"]) for r in data if r["outcome"] == outcome and r["specification"] == spec}
    return [est[name(d)] for d in ORDER]


for outcome in ["Star rating", "Would take again"]:
    print(f"{outcome}: joint weights (standardized)")
    w = {s: weights(s, outcome) for s in SCORES}
    rows_ = [dict(dim=d, **{s: w[s][i] for s in SCORES}) for i, d in enumerate(ORDER)]
    table(rows_, [("Text measure", lambda r: r["dim"])] + [(s, (lambda s: lambda r: f"{r[s]:+.3f}")(s)) for s in SCORES])
    print("(Under emotion words, the affect rows are LIWC-22 positive and negative emotion.)\n")

agree = {}
for s in SCORES:
    ws, wb = weights(s, "Star rating"), weights(s, "Would take again")
    cos = sum(a * b for a, b in zip(ws, wb)) / math.sqrt(sum(a * a for a in ws) * sum(b * b for b in wb))
    agree[s] = (pearson(ws, wb), pearson(ranks(ws), ranks(wb)), cos)
    print(f"Star vs binary joint weights, {s:14s}: r = {agree[s][0]:.3f}, Spearman = {agree[s][1]:.3f}, cosine = {agree[s][2]:.3f}")
print()
top = {s: sorted(ORDER, key=lambda d: -abs(weights(s, "Star rating")[ORDER.index(d)]))[:2] for s in SCORES}
check("Warmth and competence carry the two largest star weights under every scoring",
      all(set(t) == {"Warmth", "Competence"} for t in top.values()))
check("Star and binary joint weights agree under every scoring (r > 0.99, identical rank order)",
      all(a[0] > 0.99 and a[1] == 1.0 for a in agree.values()))


Star rating: joint weights (standardized)
Text measure          Masked (main)    Original    Emotion words  
Warmth                +0.281           +0.293      +0.318         
Competence            +0.145           +0.140      +0.185         
Brilliance            +0.035           +0.070      +0.038         
Effort                +0.092           +0.066      +0.085         
Complaint severity    -0.001           -0.022      -0.002         
Positive affect       +0.109           +0.107      +0.067         
Negative affect       -0.121           -0.119      -0.050         
(Under emotion words, the affect rows are LIWC-22 positive and negative emotion.)

Would take again: joint weights (standardized)
Text measure          Masked (main)    Original    Emotion words  
Warmth                +0.253           +0.258      +0.301         
Competence            +0.137           +0.137      +0.173         
Brilliance            +0.028           +0.064      +0.035         
Effort                +0

## Cross-format comparison

Star and would-take-again gaps estimated on the same 487,110 reviews and standardized within that sample (SI Appendix, *Cross-Format Comparison*). The 95% confidence interval for the correlation comes from 1,000 bootstrap replicates that resample instructors with replacement and re-estimate both gap models in each replicate (`si_bootstrap_correlation.csv`).

In [7]:
cc = rows("figure2_a.csv")
nonref = [r for r in cc if r["group"] != "White_Male"]
nonblack = [r for r in nonref if not r["group"].startswith("Black")]
xs = lambda d: [num(r["estimate_star"]) for r in d]
ys = lambda d: [num(r["estimate_wta"]) for r in d]
r7, r8, r5 = pearson(xs(nonref), ys(nonref)), pearson(xs(cc), ys(cc)), pearson(xs(nonblack), ys(nonblack))
ratios = [num(r["estimate_wta"]) / num(r["estimate_star"]) for r in nonref]
print(f"r across seven nonreference groups = {r7:.4f}; with White men = {r8:.4f}; without the Black groups = {r5:.4f}")
print(f"Binary gap / star gap: {min(ratios):.3f} to {max(ratios):.3f}  ->  {100 * (1 - max(ratios)):.1f}% to {100 * (1 - min(ratios)):.1f}% smaller\n")
check("r = 0.998 with or without White men", round(r7, 3) == 0.998 and round(r8, 3) == 0.998)
check("r = 0.987 without the two Black groups", round(r5, 3) == 0.987)
check("Binary gaps are 8.4 to 22.6 percent smaller than star gaps",
      round(100 * (1 - max(ratios)), 1) == 8.4 and round(100 * (1 - min(ratios)), 1) == 22.6)

boot = rows("si_bootstrap_correlation.csv")


def percentile(values, q):
    """Linear-interpolation percentile (the default in R and NumPy)."""
    v = sorted(values)
    h = (len(v) - 1) * q
    lo = math.floor(h)
    return v[lo] + (h - lo) * (v[min(lo + 1, len(v) - 1)] - v[lo])


rb = [num(r["r"]) for r in boot]
ci = (percentile(rb, 0.025), percentile(rb, 0.975))
print(f"\nInstructor bootstrap: {len(rb):,} replicates, 95% percentile CI for r = {ci[0]:.4f} to {ci[1]:.4f}")
check("Instructor-bootstrap 95% CI for r is 0.96 to 1.00", round(ci[0], 2) == 0.96 and round(ci[1], 2) == 1.00)

r across seven nonreference groups = 0.9977; with White men = 0.9983; without the Black groups = 0.9866
Binary gap / star gap: 0.774 to 0.916  ->  8.4% to 22.6% smaller

✓ r = 0.998 with or without White men
✓ r = 0.987 without the two Black groups
✓ Binary gaps are 8.4 to 22.6 percent smaller than star gaps

Instructor bootstrap: 1,000 replicates, 95% percentile CI for r = 0.9595 to 0.9986
✓ Instructor-bootstrap 95% CI for r is 0.96 to 1.00


## Sensitivity analyses

Additional star-rating gaps in sample B under each check (SI Appendix, *Sensitivity and Diagnostic Analyses*): LIWC-22 emotion words in place of tone, standard errors clustered by institution, instructor weighting, a mixed model with crossed instructor and institution intercepts, pronoun-confirmed gender, stricter surname thresholds, and separate models within the six largest disciplines.

In [8]:
sens = rows("si_sensitivity_additional_gaps.csv")
checks = []
for r in sens:
    key = (r["check"], r["text_model"])
    if key not in checks:
        checks.append(key)
cols = [g for g in GROUPS[1:]]
print(f"{'Check':58s} {'Text model':30s} " + " ".join(f"{SHORT[g][:13]:>13s}" for g in cols))
for key in checks:
    d = {r["group"]: r for r in sens if (r["check"], r["text_model"]) == key}
    print(f"{key[0][:58]:58s} {key[1][:30]:30s} " + " ".join(f"{num(d[g]['additional']):>+13.3f}" for g in cols))
print()
core = [k for k in checks if not k[0].startswith("Within discipline")]
check("Every additional star gap stays negative under every non-discipline check",
      all(num(r["additional"]) < 0 for r in sens if (r["check"], r["text_model"]) in core))
main_add = {r["group"]: num(r["additional"]) for r in sens if r["check"] == "Main specification"}
emo_add = {r["group"]: num(r["additional"]) for r in sens if r["check"].startswith("LIWC emotion")}
shift = max(abs(emo_add[g] - main_add[g]) for g in main_add)
check(f"Emotion words in place of tone change no additional gap by more than 0.005 SD (largest change {shift:.4f})", shift <= 0.005)
disc = [r for r in sens if r["check"].startswith("Within discipline")]
check(f"Within each of the six largest disciplines, all {len(disc)} additional gaps from the supervised model are negative",
      all(num(r["additional"]) < 0 for r in disc))

Check                                                      Text model                       White women     Black men   Black women  Hispanic men Hispanic wome     Asian men   Asian women
Main specification                                         Seven measures                        -0.035        -0.164        -0.167        -0.071        -0.093        -0.109        -0.107
LIWC emotion words in place of tone                        Seven measures                        -0.037        -0.165        -0.172        -0.072        -0.095        -0.109        -0.108
Standard errors clustered by institution                   Seven measures                        -0.035        -0.164        -0.167        -0.071        -0.093        -0.109        -0.107
Instructor weighting (1/n_j)                               Seven measures                        -0.035        -0.162        -0.171        -0.067        -0.091        -0.116        -0.120
Pronoun-confirmed gender                                   S

## Original versus gender-neutralized text scores

The main analyses use embedding scores computed after masking names and gendered vocabulary (SI Appendix, *Gender-Vocabulary Neutralization*). This comparison repeats the star decomposition with the original scores.

In [9]:
mo = rows("si_masked_vs_original_scores.csv")
table([r for r in mo], [("Group", lambda r: SHORT[r["group"]]),
                        ("Additional (masked)", lambda r: f"{num(r['additional_masked']):+.3f}"),
                        ("Additional (original)", lambda r: f"{num(r['additional_original']):+.3f}"),
                        ("Predicted (masked)", lambda r: f"{num(r['text_predicted_masked']):+.3f}"),
                        ("Predicted (original)", lambda r: f"{num(r['text_predicted_original']):+.3f}")])
print()
check("With original scores, every additional star gap stays negative",
      all(num(r["additional_original"]) < 0 for r in mo))

Group             Additional (masked)    Additional (original)    Predicted (masked)    Predicted (original)  
White women       -0.035                 -0.084                   +0.001                +0.050                
Black men         -0.164                 -0.161                   -0.124                -0.127                
Black women       -0.167                 -0.211                   -0.116                -0.073                
Hispanic men      -0.071                 -0.085                   -0.023                -0.009                
Hispanic women    -0.093                 -0.151                   -0.005                +0.054                
Asian men         -0.109                 -0.111                   -0.062                -0.060                
Asian women       -0.107                 -0.152                   -0.047                -0.002                

✓ With original scores, every additional star gap stays negative


## Pronoun check of inferred gender

Agreement between the gender inferred from first names and the pronouns students used for the instructor, among instructors whose reviews contain pronouns.

In [10]:
pv = rows("si_pronoun_gender_agreement.csv")
table(pv, [("Group", lambda r: SHORT[r["group"]]), ("Instructors with pronouns", lambda r: f"{int(r['instructors_with_pronouns']):,}"),
           ("Agreement", lambda r: f"{100 * num(r['agreement_share']):.1f}%")])
overall = sum(num(r["agreement_share"]) * num(r["instructors_with_pronouns"]) for r in pv) / sum(num(r["instructors_with_pronouns"]) for r in pv)
print()
check(f"Pronouns agree with the inferred gender for 98.6% of instructors (computed: {100 * overall:.1f}%)", round(100 * overall, 1) == 98.6)

Group             Instructors with pronouns    Agreement  
Asian women       8,445                        91.4%      
Asian men         11,485                       92.7%      
Black women       819                          93.0%      
Black men         1,321                        98.9%      
Hispanic women    11,319                       99.3%      
Hispanic men      12,796                       99.1%      
White women       135,652                      98.7%      
White men         181,271                      99.2%      

✓ Pronouns agree with the inferred gender for 98.6% of instructors (computed: 98.6%)


## Sentence-extraction validation

Human ratings of the instructor-directed sentence extraction (SI Appendix, *Instructor-Directed Sentence Extraction*). Sixty participants rated the same six reviews, each highlighted once by the model-based extraction and once by a dictionary method, on two 1-to-7 scales: whether the highlighting captured all relevant text and whether it avoided irrelevant text. Differences are paired on participant means, with a t-based 95% confidence interval (59 degrees of freedom), and rater agreement is ICC(2,k) across the 12 highlighted panels. The study's own files, R code, and README are in `sentence_extraction_validation/`; this cell recomputes the reported numbers from its de-identified long-format ratings.

In [11]:
VAL = ROOT / "sentence_extraction_validation" / "data"
with (VAL / "validation_long.csv").open(newline="", encoding="utf-8") as f:
    ratings = list(csv.DictReader(f))
T975_DF59 = 2.0009953780  # 97.5th percentile of Student's t with 59 degrees of freedom


def paired_difference(rs, measure):
    """Model minus dictionary, paired on participant means."""
    by = {}
    for r in rs:
        by.setdefault((r["participant"], r["method"]), []).append(float(r[measure]))
    people = sorted({p for p, _ in by})
    d = [sum(by[(p, "LLM")]) / len(by[(p, "LLM")]) - sum(by[(p, "Dictionary")]) / len(by[(p, "Dictionary")]) for p in people]
    m = sum(d) / len(d)
    se = math.sqrt(sum((x - m) ** 2 for x in d) / (len(d) - 1)) / math.sqrt(len(d))
    return m, m - T975_DF59 * se, m + T975_DF59 * se


def icc2k(measure):
    """ICC(2,k): two-way random effects, absolute agreement, mean of k raters."""
    panels = sorted({(r["review"], r["option"]) for r in ratings})
    raters = sorted({r["participant"] for r in ratings})
    v = {(r["review"], r["option"], r["participant"]): float(r[measure]) for r in ratings}
    X = [[v[(a, b, p)] for p in raters] for a, b in panels]
    n, k = len(X), len(raters)
    gm = sum(map(sum, X)) / (n * k)
    rm = [sum(x) / k for x in X]
    cm = [sum(X[i][j] for i in range(n)) / n for j in range(k)]
    msr = k * sum((a - gm) ** 2 for a in rm) / (n - 1)
    msc = n * sum((b - gm) ** 2 for b in cm) / (k - 1)
    mse = sum((X[i][j] - rm[i] - cm[j] + gm) ** 2 for i in range(n) for j in range(k)) / ((n - 1) * (k - 1))
    return (msr - mse) / (msr + (msc - mse) / n)


print(f"{len({r['participant'] for r in ratings})} participants, {len({r['review'] for r in ratings})} reviews, {len(ratings)} rating rows\n")
res = {}
for label, rs in [("All six reviews", ratings), ("Instructor reviews", [r for r in ratings if r["domain"] == "Professor"])]:
    for measure in ["capture", "avoid"]:
        res[(label, measure)] = paired_difference(rs, measure)
table([dict(s=k[0], m=k[1], d=v) for k, v in res.items()],
      [("Sample", lambda r: r["s"]), ("Measure", lambda r: r["m"]),
       ("Model minus dictionary", lambda r: f"{r['d'][0]:+.2f}"),
       ("95% CI", lambda r: f"[{r['d'][1]:.2f}, {r['d'][2]:.2f}]")])
icc = {m: icc2k(m) for m in ["capture", "avoid"]}
print(f"\nICC(2,k): capture {icc['capture']:.3f}, avoid {icc['avoid']:.3f}\n")

r2 = lambda x: round(x, 2)
check("All six reviews: +1.05 [0.74, 1.37] on capture and +1.18 [0.90, 1.47] on avoid",
      tuple(map(r2, res[("All six reviews", "capture")])) == (1.05, 0.74, 1.37)
      and tuple(map(r2, res[("All six reviews", "avoid")])) == (1.18, 0.90, 1.47))
check("Instructor reviews: +1.66 on capture and +1.48 on avoid",
      r2(res[("Instructor reviews", "capture")][0]) == 1.66 and r2(res[("Instructor reviews", "avoid")][0]) == 1.48)
check("Agreement across raters: ICC(2,k) = 0.94 and 0.96", r2(icc["capture"]) == 0.94 and r2(icc["avoid"]) == 0.96)

60 participants, 6 reviews, 720 rating rows

Sample                Measure    Model minus dictionary    95% CI        
All six reviews       capture    +1.05                     [0.74, 1.37]  
All six reviews       avoid      +1.18                     [0.90, 1.47]  
Instructor reviews    capture    +1.66                     [1.25, 2.06]  
Instructor reviews    avoid      +1.48                     [1.06, 1.89]  

ICC(2,k): capture 0.936, avoid 0.958

✓ All six reviews: +1.05 [0.74, 1.37] on capture and +1.18 [0.90, 1.47] on avoid
✓ Instructor reviews: +1.66 on capture and +1.48 on avoid
✓ Agreement across raters: ICC(2,k) = 0.94 and 0.96
